# Multi-Node Heat Network

Two buildings with independent heat networks, served by a shared gas grid
but separate boilers.

**New concepts**: Multi-node carriers (`node` parameter) · Independent balance equations · Spatial separation

In [ ]:
import math
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from fluxopt import Converter, Effect, Flow, Port, optimize

pio.renderers.default = 'notebook_connected'

## Input Data — 24 h with different demand profiles

Building A (office): daytime peak. Building B (hospital): constant baseload with evening peak.

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]

# Office: peaks during work hours
demand_a = [20 + 60 * max(0.0, math.sin(math.pi * ((h % 24) - 6) / 12)) for h in range(n)]
# Hospital: constant 40 kW base + evening peak
demand_b = [40 + 30 * max(0.0, math.sin(math.pi * ((h % 24) - 14) / 8)) for h in range(n)]

# Time-varying gas price
gas_price = [0.04 if (h % 24) < 6 or (h % 24) >= 22 else 0.07 for h in range(n)]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.15,
    subplot_titles=('Heat Demand by Building', 'Gas Price'),
)
fig.add_trace(go.Bar(x=timesteps, y=demand_a, name='Building A (office)', marker_color='#ef553b'), row=1, col=1)
fig.add_trace(go.Bar(x=timesteps, y=demand_b, name='Building B (hospital)', marker_color='#ab63fa'), row=1, col=1)
fig.add_trace(
    go.Scatter(x=timesteps, y=gas_price, line_shape='hv', line_color='#636efa', showlegend=False),
    row=2,
    col=1,
)
fig.update_layout(height=400, margin={'l': 50, 'r': 20, 't': 30, 'b': 20}, template='plotly_white', barmode='group')
fig.update_yaxes(title_text='kW', row=1, col=1)
fig.update_yaxes(title_text='EUR/kWh', row=2, col=1)
fig

## System

Both buildings share a gas carrier but have **independent heat networks**:

```
                    ┌── Boiler A η=90% ──► [heat:A] ──► Office
Gas Grid ──► [gas] ─┤
                    └── Boiler B η=95% ──► [heat:B] ──► Hospital
```

The `node` parameter on each heat flow creates separate balance equations.
Heat from boiler A cannot serve building B, and vice versa.

In [ ]:
max_a = max(demand_a)
max_b = max(demand_b)

result = optimize(
    timesteps=timesteps,
    effects=[Effect('cost', unit='EUR', is_objective=True)],
    ports=[
        # Shared gas supply
        Port(
            'gas_grid',
            imports=[Flow('gas', size=500, effects_per_flow_hour={'cost': gas_price})],
        ),
        # Building A demand (heat node A)
        Port(
            'office',
            exports=[Flow('heat', node='A', size=max_a, fixed_relative_profile=[d / max_a for d in demand_a])],
        ),
        # Building B demand (heat node B)
        Port(
            'hospital',
            exports=[Flow('heat', node='B', size=max_b, fixed_relative_profile=[d / max_b for d in demand_b])],
        ),
    ],
    converters=[
        # Boiler A: serves heat node A only
        Converter.boiler(
            'boiler_a',
            thermal_efficiency=0.90,
            fuel_flow=Flow('gas', size=200),
            thermal_flow=Flow('heat', node='A', size=100),
        ),
        # Boiler B: serves heat node B only
        Converter.boiler(
            'boiler_b',
            thermal_efficiency=0.95,
            fuel_flow=Flow('gas', size=200),
            thermal_flow=Flow('heat', node='B', size=100),
        ),
    ],
)

## Results

In [ ]:
total_heat = sum(demand_a) + sum(demand_b)
print(f'Total cost: {result.objective:.2f} EUR  |  Avg: {result.objective / total_heat * 100:.2f} ct/kWh')

In [ ]:
times = result.flow_rates.coords['time'].values

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=('Heat Supply — Node A (Office)', 'Heat Supply — Node B (Hospital)'),
)

# Node A flows
fig.add_trace(
    go.Scatter(
        x=times,
        y=result.flow_rate('boiler_a(heat:A)').values,
        name='boiler_a → heat:A',
        line_shape='hv',
        line_color='#ef553b',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=result.flow_rate('office(heat:A)').values,
        name='office demand',
        line_shape='hv',
        line_color='#ef553b',
        line_dash='dot',
    ),
    row=1,
    col=1,
)

# Node B flows
fig.add_trace(
    go.Scatter(
        x=times,
        y=result.flow_rate('boiler_b(heat:B)').values,
        name='boiler_b → heat:B',
        line_shape='hv',
        line_color='#ab63fa',
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=result.flow_rate('hospital(heat:B)').values,
        name='hospital demand',
        line_shape='hv',
        line_color='#ab63fa',
        line_dash='dot',
    ),
    row=2,
    col=1,
)

fig.update_layout(height=450, margin={'l': 50, 'r': 20, 't': 30, 'b': 20}, template='plotly_white')
fig.update_yaxes(title_text='kW', row=1, col=1)
fig.update_yaxes(title_text='kW', row=2, col=1)
fig

In [ ]:
fig = go.Figure()

for fid in result.flow_rates.coords['flow'].values:
    if 'gas' in str(fid):
        fig.add_trace(go.Scatter(x=times, y=result.flow_rate(str(fid)).values, name=str(fid), line_shape='hv'))

fig.update_layout(
    title='Gas Consumption by Boiler',
    height=300,
    margin={'l': 50, 'r': 20, 't': 40, 'b': 20},
    template='plotly_white',
)
fig.update_yaxes(title_text='kW')
fig

## Insights

- **Independent networks**: boiler A only serves the office, boiler B only serves the hospital — no cross-flow
- **Shared gas**: both boilers draw from the same gas carrier, so the gas balance sees their combined consumption
- **Different efficiencies**: boiler B (η=95%) uses less gas per kWh of heat than boiler A (η=90%)
- **Use case**: district heating with separate building loops, or multi-site industrial plants with independent thermal networks